# JAM Knowledge Mutation 001 Failure Diagnostic

Post-hoc forward-only diagnostic over the already-published JAM001 formal mutation artifacts. This does not retrain the model and cannot change the frozen upstream decision `JAM_KNOWLEDGE_MUTATION_NOT_SUPPORTED`.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/jam-knowledge-mutation-001-failure-diagnostic'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], cwd=ROOT, check=True)
print({'head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()})

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.0.0', 'huggingface_hub==1.11.0', 'safetensors==0.7.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/research/jam_knowledge_mutation_001_failure_diagnostic/validate_sources.py'], cwd=ROOT, check=True)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
    try:
        os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    except Exception:
        pass
except Exception as exc:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is required') from exc

subprocess.run([
    sys.executable, 'scripts/research/jam_knowledge_mutation_001_failure_diagnostic/publish.py',
    '--branch', BRANCH, '--preflight-only'
], cwd=ROOT, check=True)

In [ ]:
import torch, transformers, huggingface_hub
assert torch.cuda.is_available(), 'CUDA is required for the diagnostic forward pass'
free_mb, _total_mb = torch.cuda.mem_get_info(0)
free_mb //= 1024 * 1024
assert free_mb >= 12000, f'need >= 12000 MiB free GPU memory, found {free_mb}'
plan = json.loads((ROOT / 'research/validations/jam-knowledge-mutation-001-failure-diagnostic/diagnostic_plan.json').read_text())
print({
    'gpu': torch.cuda.get_device_name(0),
    'free_mb': free_mb,
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'huggingface_hub': huggingface_hub.__version__,
    'upstream_decision': plan['upstream']['formal_decision'],
    'seeds': plan['upstream']['formal_seeds'],
    'capacities': plan['upstream']['capacities'],
})

In [ ]:
def run_compact(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(
            command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env={**os.environ, 'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'TRANSFORMERS_NO_ADVISORY_WARNINGS': '1'}
        )
        assert process.stdout is not None
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            if line.startswith('[jam001diag]'):
                print(line, end='')
        returncode = process.wait()
    if returncode != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]
        print('\n=== Diagnostic log tail ===')
        print('\n'.join(tail))
        subprocess.run(['nvidia-smi'], check=False)
        raise RuntimeError(f'diagnostic failed with exit code {returncode}; full log: {log_path}')

artifact = ROOT / 'artifacts/experiments/jam-knowledge-mutation-001-failure-diagnostic/diagnostic.json'
if artifact.is_file():
    print('[jam001diag] diagnostic already published; skipping GPU forward pass')
else:
    run_compact([
        sys.executable, 'scripts/research/jam_knowledge_mutation_001_failure_diagnostic/run_diagnostic.py',
        '--device', 'cuda:0'
    ], ROOT / 'results/jam-knowledge-mutation-001-failure-diagnostic-launcher/run.log')
    subprocess.run([
        sys.executable, 'scripts/research/jam_knowledge_mutation_001_failure_diagnostic/publish.py',
        '--branch', BRANCH
    ], cwd=ROOT, check=True)

In [ ]:
result_path = ROOT / 'artifacts/experiments/jam-knowledge-mutation-001-failure-diagnostic/diagnostic.json'
result = json.loads(result_path.read_text())
print(json.dumps({
    'status': result['status'],
    'classification': result['interpretation']['classification'],
    'upstream_formal_decision_unchanged': result['upstream_formal_decision_unchanged'],
    'capacity4_full_gains': result['interpretation']['capacity4_full_gains'],
    'capacity4_content_plus_eos_gains': result['interpretation']['capacity4_content_plus_eos_gains'],
    'capacity4_prefix_gains': result['interpretation']['capacity4_prefix_gains'],
    'capacity4_training_style_full_gains': result['interpretation']['capacity4_training_style_full_gains'],
}, indent=2))

The diagnostic is intentionally post-hoc and forward-only. Its classification is descriptive and cannot replace or revise the frozen JAM001 formal decision.